In [12]:
import requests
import os
import sys
import time
from dotenv import load_dotenv
import logging as log
import json
import pandas as pd
import pandas_ta as ta

from trading_scripts.api import utils
from trading_scripts.api.login import LoginManager
from trading_scripts.api.open_positions import OpenPositionsAPI
from trading_scripts.api.price_data import PriceData


In [13]:
# Configure logging to output to the notebook's standard output
log.basicConfig(level=log.DEBUG, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = log.getLogger()
logger.addHandler(log.StreamHandler(sys.stdout))

In [14]:
login_manager = LoginManager()

In [ ]:
login_manager.login()

print(f"Auth: {login_manager.auth_trading_api}")
print(f"Cookie: {login_manager.cookie}")
print(f"UUID: {login_manager.system_uuid}")
print(f"RT Token: {login_manager.rt_token}")

auth_trading_api = login_manager.auth_trading_api
cookie = login_manager.cookie
UUID = login_manager.system_uuid
rt_token = login_manager.rt_token


In [54]:
from urllib.parse import urlparse, parse_qs

def refresh_token():
    """Refresh the authentication token."""
    url = f"{utils.PLATFORM_URL}/mtr-backend/refresh-token"
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
        "Origin": "https://platform.citytradersimperium.com",
        "Referer": "https://platform.citytradersimperium.com/dashboard",
        "Accept-Encoding": "gzip, deflate, br",
        "Accept-Language": "en-US,en;q=0.8",
        "Sec-CH-UA": '"Brave";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
        "Sec-CH-UA-Mobile": "?0",
        "Sec-CH-UA-Platform": "Windows",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "Sec-GPC": "1",
    }

    session = requests.Session()
    session.headers.update(headers)
    session.cookies.set("rt", rt_token, domain="citytradersimperium.com", path="/mtr-backend/refresh-token")
    try:
        response = session.post(url, json={})
        response.raise_for_status()
        print("Response status code:", response.status_code)
        print("Response headers:", response.headers)
        print("Response cookies:", response.cookies)
        print("set-cookie header:", response.headers.get("Set-Cookie"))

        # Extract the updated co-auth token from cookies
        co_auth = next(
                    (
                        cookie.value
                        for cookie in response.cookies
                        if cookie.name == "co-auth"
                    ),
                    None,
                )
        new_rt_token = next(
                    (
                        cookie.value
                        for cookie in response.cookies
                        if cookie.name == "rt" and "mtr-backend" in cookie.path
                    ),
                    None,
                )
        if co_auth and rt_token:
            print(f"Token refreshed successfully. New co-auth token: {co_auth}")
            print(f"Token refreshed successfully. New rt token: {new_rt_token}")
            return co_auth, new_rt_token
        else:
            print("Failed to refresh token: No co-auth token in response.")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Failed to refresh token: {e}")
        return None

In [ ]:
refresh_token()

In [15]:
def get_symbol_data(login_manager, symbol):
    """Get the latest news for a symbol."""
    url = f"{utils.PLATFORM_URL}/market-data-api/{login_manager.system_uuid}/api/trading-view/symbols?symbol={symbol}"
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
        "Origin": "https://platform.citytradersimperium.com",
        "Referer": "https://platform.citytradersimperium.com/dashboard",
        "Accept-Encoding": "gzip, deflate, br",
        "Accept-Language": "en-US,en;q=0.8",
        "Sec-CH-UA": '"Brave";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
        "Sec-CH-UA-Mobile": "?0",
        "Sec-CH-UA-Platform": "Windows",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "Sec-GPC": "1",
    }

    session = requests.Session()
    session.headers.update(headers)
    session.cookies.set("co-auth", cookie, domain="citytradersimperium.com", path="/mtr-backend/refresh-token")
    try:
        response = session.get(url)
        response.raise_for_status()
        data = response.json()
        return data
    except requests.exceptions.RequestException as e:
        print(f"Failed to get symbol data: {e}")
        return None

In [ ]:
symbols = ["EURUSD", "BTCUSDC", "US500", "US30"]
for symbol in symbols:
    data = get_symbol_data(login_manager, symbol)
    print(f"Symbol: {symbol}: {json.dumps(data, indent=4)}")

In [58]:
def get_balance(login_manager):
    """Get the account balance information."""
    url = f"{utils.PLATFORM_URL}/mtr-api/{login_manager.system_uuid}/balance"
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
        "Origin": f"{utils.PLATFORM_URL}",
        "Referer": f"{utils.PLATFORM_URL}/dashboard",
        "Auth-trading-api": login_manager.auth_trading_api,
        "Cookie": f"co-auth={login_manager.cookie}",
        "Accept-Encoding": "gzip, deflate, br",
        "Accept-Language": "en-US,en;q=0.8",
        "Sec-CH-UA": '"Brave";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
        "Sec-CH-UA-Mobile": "?0",
        "Sec-CH-UA-Platform": "Windows",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "Sec-GPC": "1",
    }

    session = requests.Session()
    session.headers.update(headers)
    try:
        response = session.get(url)
        response.raise_for_status()
        data = response.json()
        return data
    except requests.exceptions.RequestException as e:
        print(f"Failed to get balance data: {e}")
        return None

In [ ]:
get_balance(login_manager)

In [ ]:
utils.fetch_open_positions_once(login_manager)

In [ ]:
price_data = PriceData(login_manager)
historical_data = price_data.fetch_market_data("EURUSD", 5, 5000)
df = pd.DataFrame(historical_data)
df.tail()

In [11]:
def calc_linear_regression(symbol, timeframe):
    """Calculate the linear regression for the given symbol and timeframe.

    Args:
        symbol (str): The trading symbol.
        timeframe (str): The timeframe for the linear regression.
    """
    price_data = PriceData(LoginManager())
    ohlc = price_data.fetch_market_data(symbol, timeframe)
    df = pd.DataFrame(ohlc)
    df.set_index("t", inplace=True)
    df.ta.linreg(close="c", length=14, append=True)
    return df["LR_14"].iloc[-1] - df["LR_14"].iloc[-2]

In [ ]:
linear_reg_1H = calc_linear_regression("BTCUSDC", 60)
linear_reg_15M = calc_linear_regression("BTCUSDC", 15)

print(f"Linear regression 1H: {linear_reg_1H}")
print(f"Linear regression 15M: {linear_reg_15M}")

In [ ]:
open_positions = utils.fetch_open_positions_once(login_manager)
print(json.dumps(open_positions, indent=4))

In [18]:
price_data = PriceData()
symbol = "BTCUSDC"
ohlc = price_data.fetch_market_data(login_manager, symbol, 5)
df = pd.DataFrame(ohlc).set_index("t")
candles = df.ta.cdl_pattern(open="o", high="h", low="l", close="c", name=["engulfing", "shootingstar", "hammer"])
candles.tail()

2024-12-30 10:47:33,441 - root - INFO - Fetching market data for symbol: BTCUSDC, timeframe 5


Fetching market data for symbol: BTCUSDC, timeframe 5
Fetching market data for symbol: BTCUSDC, timeframe 5


2024-12-30 10:47:33,443 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): platform.citytradersimperium.com:443


Starting new HTTPS connection (1): platform.citytradersimperium.com:443
Starting new HTTPS connection (1): platform.citytradersimperium.com:443


2024-12-30 10:47:33,815 - urllib3.connectionpool - DEBUG - https://platform.citytradersimperium.com:443 "GET /market-data-api/82b821d8-eeff-4352-ae2a-753c1daf8546/api/trading-view/history?symbol=BTCUSDC&resolution=5&from=1735423653&to=1735573653&shouldRetrieveOnlyFromCache=true&countback=500 HTTP/11" 200 None


https://platform.citytradersimperium.com:443 "GET /market-data-api/82b821d8-eeff-4352-ae2a-753c1daf8546/api/trading-view/history?symbol=BTCUSDC&resolution=5&from=1735423653&to=1735573653&shouldRetrieveOnlyFromCache=true&countback=500 HTTP/11" 200 None
https://platform.citytradersimperium.com:443 "GET /market-data-api/82b821d8-eeff-4352-ae2a-753c1daf8546/api/trading-view/history?symbol=BTCUSDC&resolution=5&from=1735423653&to=1735573653&shouldRetrieveOnlyFromCache=true&countback=500 HTTP/11" 200 None
[X] Please install TA-Lib to use engulfing. (pip install TA-Lib)
[X] Please install TA-Lib to use shootingstar. (pip install TA-Lib)
[X] Please install TA-Lib to use hammer. (pip install TA-Lib)


,o,h,l,c,s
t,,,,,
1735572300000,91561.6,91776.1,91501.0,91646.2,ok
1735572600000,91649.0,91790.9,91595.4,91749.0,ok
1735572900000,91723.9,91978.9,91591.0,91978.9,ok
1735573200000,91964.5,92179.9,91943.5,92134.9,ok
1735573500000,92142.3,92427.7,92083.2,92095.1,ok


In [ ]:
price_data = PriceData()
symbol = "EURUSD"
market_watch = price_data.fetch_market_watch(login_manager, symbol)
print(symbol, json.dumps(market_watch, indent=4))